# Gestión manual de registros — `form_entries.db`

Utilidades para administrar la tabla `processed_entries` de la base de datos de deduplicación.

- La tabla tiene una sola columna: `entry_id` (INTEGER PRIMARY KEY).
- Cada `entry_id` corresponde a una OT ya procesada por el pipeline.
- **Agregar** un `entry_id` hace que el pipeline lo considere ya procesado (lo omitirá).
- **Remover** un `entry_id` hace que el pipeline lo vuelva a procesar en la próxima corrida.

Ejecuta primero la celda de configuración.

In [ ]:
# === Configuración ===
import os
import sqlite3

# Ruta a la base de datos (misma carpeta que este notebook)
BASE_DIR = os.path.dirname(os.path.abspath('gestion_registros.ipynb'))
DB_PATH = os.path.join(BASE_DIR, 'form_entries.db')

print(f'Base de datos: {DB_PATH}')
print(f'Existe: {os.path.exists(DB_PATH)}')


def listar_registros():
    """Devuelve y muestra todos los entry_id almacenados, ordenados."""
    with sqlite3.connect(DB_PATH) as con:
        ids = [row[0] for row in con.execute(
            'SELECT entry_id FROM processed_entries ORDER BY entry_id'
        )]
    print(f'Total de registros: {len(ids)}')
    print(ids)
    return ids


listar_registros()

## Módulo 1 — Agregar un registro

Inserta un `entry_id` en `processed_entries`. Usa `INSERT OR IGNORE`, por lo que repetir un ID existente no genera error ni duplicados.

In [ ]:
# === Módulo 1: Agregar un registro ===
def agregar_registro(entry_id: int) -> bool:
    """Agrega un entry_id a la tabla processed_entries.

    Devuelve True si se insertó una fila nueva, False si ya existía.
    """
    entry_id = int(entry_id)
    with sqlite3.connect(DB_PATH) as con:
        cur = con.execute(
            'INSERT OR IGNORE INTO processed_entries (entry_id) VALUES (?)',
            (entry_id,),
        )
        con.commit()
        if cur.rowcount > 0:
            print(f'ID {entry_id} agregado correctamente.')
            return True
        print(f'ID {entry_id} ya existía; no se realizaron cambios.')
        return False


# --- Uso: cambia el valor por el entry_id que quieras agregar ---
ENTRY_ID_A_AGREGAR = 999
agregar_registro(ENTRY_ID_A_AGREGAR)
listar_registros()

## Módulo 2 — Remover un registro

Elimina un `entry_id` de `processed_entries`. Si el ID no existe, lo informa sin generar error.

In [ ]:
# === Módulo 2: Remover un registro ===
def remover_registro(entry_id: int) -> bool:
    """Elimina un entry_id de la tabla processed_entries.

    Devuelve True si se eliminó una fila, False si no existía.
    """
    entry_id = int(entry_id)
    with sqlite3.connect(DB_PATH) as con:
        cur = con.execute(
            'DELETE FROM processed_entries WHERE entry_id = ?',
            (entry_id,),
        )
        con.commit()
        if cur.rowcount > 0:
            print(f'ID {entry_id} removido correctamente.')
            return True
        print(f'ID {entry_id} no existía; no se realizaron cambios.')
        return False


# --- Uso: cambia el valor por el entry_id que quieras remover ---
ENTRY_ID_A_REMOVER = 999
remover_registro(ENTRY_ID_A_REMOVER)
listar_registros()